In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import pickle as pkl

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import roc_auc_score

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
from sklearn.metrics import roc_auc_score

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, ShuffleSplit, StratifiedShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder

skf = StratifiedKFold(5, random_state = 123, shuffle = True)
ss = StratifiedShuffleSplit(1, train_size = 0.8, random_state = 123)
ss_v = StratifiedShuffleSplit(1, train_size = 0.9, random_state = 123)

In [ ]:
from mllabs.processor import PolarsLoader, ExprProcessor, PandasConverter

In [ ]:
p = make_pipeline(
    PolarsLoader(predefined_types={'id': pl.Int64}),
    ExprProcessor({
        'loan_paid_back': pl.col('loan_paid_back').cast(pl.Int8)
    }),
    PandasConverter(index_col = 'id')
)
df_train = p.fit_transform('data/train.csv')
df_test = p.transform('data/test.csv')

In [ ]:
X_all = df_test.columns.tolist()
X_num = ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
X_cat = ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose']
y = 'loan_paid_back'

In [ ]:
# import shutil
# shutil.rmtree('exp/exp1')
import os

In [ ]:
from mllabs import Experimenter
from mllabs.collector import MetricCollector, ModelAttrCollector, StackingCollector
from mllabs import Connector
from sklearn.metrics import roc_auc_score

if os.path.exists('exp/exp1'):
    e = Experimenter.load('exp/exp1', df_train)
else:
    e = Experimenter.create(
        df_train, 'exp/exp1', sp = StratifiedShuffleSplit(n_splits=1, random_state = 123), 
        sp_v = StratifiedShuffleSplit(n_splits=1, train_size=0.9, random_state = 123), splitter_params = {'y': y}
    )

In [ ]:
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression

# Configuration

In [ ]:
e.add_collector(
    MetricCollector(
        'AUC', Connector(edges = {'y': [(None, y)]}), '.*'+ y +'_1', roc_auc_score, include_train = True
    )
)
e.add_collector(
    ModelAttrCollector(
        'lgb_feature_importance', Connector(processor = lgb.LGBMClassifier), 'feature_importances'
    )
)
e.add_collector(
    StackingCollector(
        'stacking', Connector(edges = {'y': [(None, y)]}),
        '.*' + y + '_1', method='mean', include_target=True
    )
)

e.set_grp('clf', role = 'head', method = 'predict_proba', edges = {'y': [(None, y)]})
e.set_grp('lgb', parent = 'clf', processor = lgb.LGBMClassifier, params={'verbose': -1, 'early_stopping': lgb.early_stopping(100), 'eval_metric': 'AUC'})
e.set_grp('xgb', parent = 'clf', processor = xgb.XGBClassifier)
e.set_grp('cb', parent = 'clf', processor = cb.CatBoostClassifier)
e.set_grp('lr', parent = 'clf', processor = LogisticRegression)
e.set_grp('pre', role = 'stage', method = 'transform')

## Stage Nodes

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
e.set_node(
    'ord', grp = 'pre', processor=OrdinalEncoder, 
    edges ={'X': [(None, 'grade_subgrade')]}, params={'categories': [np.sort(df_train['grade_subgrade'].unique())]}
)

e.set_node(
    'ohe', grp = 'pre', processor=OneHotEncoder, 
    edges ={'X': [(None, X_cat)]}, params={'sparse_output': False}
)

e.set_node(
    'std', grp = 'pre', processor=StandardScaler, edges ={'X': [(None, X_num)]}
)
e.build()

## LightGBM

In [ ]:
e.set_node('lgb1', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.1})
e.exp()

In [ ]:
e.set_node(
    'lgb2', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.05}
)
e.exp()

In [ ]:
e.add_collector(
    ModelAttrCollector(
        'lgb_evals_results', 
        Connector(processor=lgb.LGBMClassifier),
        'evals_result'
    )
)

In [ ]:
e.set_node(
    'lgb3', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.2}
)
e.exp()

In [ ]:
e.set_node(
    'lgb4', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.075}
)
e.exp()

In [ ]:
e.set_node(
    'lgb5', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.075, 'num_leaves': 15}
)

In [ ]:
e.set_node(
    'lgb6', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.1, 'num_leaves': 15}
)
e.exp()

In [ ]:
e.set_node(
    'lgb7', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.05, 'num_leaves': 15}
)
e.exp()

In [ ]:
e.set_node(
    'lgb8', grp = 'lgb', edges = {'X': [(None, X_cat)]}, params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.05, 'num_leaves': 15}
)
e.exp()

In [ ]:
e.set_node(
    'lgb9', grp = 'lgb', edges = {'X': [(None, X_num + X_cat)]}, 
    params = {'categorical_features': X_cat, 'n_estimators': 10000, 'learning_rate': 0.075, 'num_leaves': 63}
)
e.exp()

In [ ]:
d = e.collectors['lgb_evals_results'].get_attrs('lgb')
s_best_iteration = pd.Series({
    k: np.mean(
        [[j.unstack().unstack()[('valid_1', 'auc')].argmax() for j in i] for i in v]
    )
    for k, v in d.items()
    }, name = ('evals_result', 'best_iteration'))
e.pipeline.compare_nodes(
    e.pipeline.get_node_names('lgb*')
)['LGBMClassifier'].fillna('default').join(
    e.collectors['AUC'].get_metrics_agg()[0].stack().rename('AUC').to_frame().unstack()
).join(
    s_best_iteration
).sort_values(('AUC', 'valid'), ascending = False)

## XGB

In [ ]:
e.add_collector(
    ModelAttrCollector('xgb_feature_importance', Connector(processor=xgb.XGBClassifier), 'feature_importances', params={'importance_type': 'gain'})
)
e.add_collector(
    ModelAttrCollector('xgb_evals_results', Connector(processor=xgb.XGBClassifier), 'evals_result')
)

# XGB with preprocessed stage features (ohe + std + ord)
e.set_node('xgb1', grp='xgb', edges={'X': [('ohe', None), ('std', None), ('ord', None)]},
    params={'n_estimators': 10000, 'learning_rate': 0.1, 'early_stopping_rounds': 100, 'eval_metric': 'auc'})
e.set_node('xgb2', grp='xgb', edges={'X': [('ohe', None), ('std', None), ('ord', None)]},
    params={'n_estimators': 10000, 'learning_rate': 0.05, 'early_stopping_rounds': 100, 'eval_metric': 'auc'})
e.set_node('xgb3', grp='xgb', edges={'X': [('ohe', None), ('std', None), ('ord', None)]},
    params={'n_estimators': 10000, 'learning_rate': 0.1, 'early_stopping_rounds': 100, 'eval_metric': 'auc', 'max_depth': 4})
e.set_node('xgb4', grp='xgb', edges={'X': [('ohe', None), ('std', None), ('ord', None)]},
    params={'n_estimators': 10000, 'learning_rate': 0.075, 'early_stopping_rounds': 100, 'eval_metric': 'auc'})
# xgb5: ohe 제외 (categorical feature 없이)
e.set_node('xgb5', grp='xgb', edges={'X': [('std', None), ('ord', None)]},
    params={'n_estimators': 10000, 'learning_rate': 0.1, 'early_stopping_rounds': 100, 'eval_metric': 'auc'})
e.exp()

In [ ]:
d = e.collectors['xgb_evals_results'].get_attrs('xgb')
s_best_iteration = pd.Series({
    k: np.mean(
        [[j.unstack().unstack()[('validation_1', 'auc')].argmax() for j in i] for i in v]
    )
    for k, v in d.items()
}, name=('evals_result', 'best_iteration'))

e.pipeline.compare_nodes(
    e.pipeline.get_node_names('xgb*')
)['XGBClassifier'].fillna('default').join(
    e.collectors['AUC'].get_metrics_agg()[0].stack().rename('AUC').to_frame().unstack()
).join(
    s_best_iteration
).sort_values(('AUC', 'valid'), ascending=False)

## Catboost

In [ ]:
e.add_collector(
    ModelAttrCollector('cb_feature_importance', Connector(processor=cb.CatBoostClassifier), 'feature_importances_pvc')
)
e.add_collector(
    ModelAttrCollector('cb_evals_results', Connector(processor=cb.CatBoostClassifier), 'evals_result')
)

# CB with raw features (native categorical handling)
e.set_node('cb1', grp='cb', edges={'X': [(None, X_num + X_cat)]},
    params={'cat_features': X_cat, 'iterations': 10000, 'learning_rate': 0.1, 'early_stopping_rounds': 100, 'eval_metric': 'AUC'})
e.set_node('cb2', grp='cb', edges={'X': [(None, X_num + X_cat)]},
    params={'cat_features': X_cat, 'iterations': 10000, 'learning_rate': 0.05, 'early_stopping_rounds': 100, 'eval_metric': 'AUC'})
e.set_node('cb3', grp='cb', edges={'X': [(None, X_num + X_cat)]},
    params={'cat_features': X_cat, 'iterations': 10000, 'learning_rate': 0.1, 'depth': 4, 'early_stopping_rounds': 100, 'eval_metric': 'AUC'})
e.set_node('cb4', grp='cb', edges={'X': [(None, X_num + X_cat)]},
    params={'cat_features': X_cat, 'iterations': 10000, 'learning_rate': 0.075, 'early_stopping_rounds': 100, 'eval_metric': 'AUC'})
# cb5: grade_subgrade 추가
e.set_node('cb5', grp='cb', edges={'X': [(None, X_num + X_cat + ['grade_subgrade'])]},
    params={'cat_features': X_cat + ['grade_subgrade'], 'iterations': 10000, 'learning_rate': 0.1, 'early_stopping_rounds': 100, 'eval_metric': 'AUC'})
e.exp()

In [ ]:
d = e.collectors['cb_evals_results'].get_attrs('cb')
s_best_iteration = pd.Series({
    k: np.mean(
        [[j.unstack().unstack()[('validation_1', 'AUC')].argmax() for j in i] for i in v]
    )
    for k, v in d.items()
}, name=('evals_result', 'best_iteration'))

e.pipeline.compare_nodes(
    e.pipeline.get_node_names('cb*')
)['CatBoostClassifier'].fillna('default').join(
    e.collectors['AUC'].get_metrics_agg()[0].stack().rename('AUC').to_frame().unstack()
).join(
    s_best_iteration
).sort_values(('AUC', 'valid'), ascending=False)

## Logistic Regression

In [ ]:
from mllabs import col

In [ ]:
for i, C in enumerate([1e-3, 1e-2, 1e-1, 1, 1e1, 1e2, 1e3]):
    e.set_node(f'lr{i}', grp='lr', edges={'X': [('std', None), ('ohe', col.ohe_drop_first)]}, params={'C': C})
e.exp()

In [ ]:
e.pipeline.compare_nodes(
    e.pipeline.get_node_names('lr*')
)['LogisticRegression'].fillna('default').join(
    e.collectors['AUC'].get_metrics_agg()[0].stack().rename('AUC').to_frame().unstack()
)

In [ ]:
from IPython.display import Markdown
Markdown(
    e.desc_node('lr1', show_params=True)
)

In [ ]:
Markdown(
    e.desc_pipeline(max_depth = 2)
)

In [ ]:
Markdown(
    e.desc_status()
)

## Stacking

In [ ]:
stk_nodes = ['lgb5', 'lgb6', 'xgb3', 'cb3', 'cb4']
df_stk = e.collectors['stacking'].get_dataset(e, stk_nodes)
stk_features = [f'{n}_pred' for n in stk_nodes]
df_stk.columns = stk_features + [y]
df_stk

In [ ]:
df_stk[stk_features].corr()

In [ ]:
if os.path.exists('exp/stk1'):
    e2 = Experimenter.load('exp/stk1', df_stk)
else:
    e2 = Experimenter.create(
        df_stk, 'exp/stk1',
        sp = StratifiedKFold(5, shuffle=True, random_state=456),
        splitter_params = {'y': y}
    )

e2.add_collector(
    MetricCollector(
        'AUC', Connector(edges = {'y': [(None, y)]}),
        '.*' + y + '_1', roc_auc_score, include_train = True
    )
)

e2.set_grp('meta', role = 'head', method = 'predict_proba', edges = {'y': [(None, y)]})
e2.set_grp('lr', parent = 'meta', processor = LogisticRegression)

for i, C in enumerate([1e-3, 1e-2, 1e-1, 1, 10]):
    e2.set_node(f'lr{i}', grp='lr', edges = {'X': [(None, stk_features)]}, params = {'C': C})
e2.exp()

In [ ]:
e2.pipeline.compare_nodes(
    e2.pipeline.get_node_names('lr*')
)['LogisticRegression'].fillna('default').join(
    e2.collectors['AUC'].get_metrics_agg()[0].stack().rename('AUC').to_frame().unstack()
).sort_values(('AUC', 'valid'), ascending = False)

# Finalize ALl Experiments Objects

In [ ]:
e.close_exp()

In [ ]:
e2.close_exp()

# Train and Predict

In [ ]:
e.add_trainer('trainer')

In [ ]:
e.trainers['trainer'].select_head(stk_nodes)

In [ ]:
e.trainers['trainer'].train()

In [ ]:
e.trainers['trainer'].node_objs['lgb5'].status

In [ ]:
e.trainers['trainer'].split_indices

In [ ]:
next(e.trainers['trainer'].node_objs['cb3'].get_obj())

In [ ]:
for i in e.trainers['trainer'].get_node_output('cb3'):
    print(i[0].data)

In [ ]:
for i in e.trainers['trainer'].process(df_test):
    print(i.data)

In [ ]:
e.trainers['trainer'].node_objs['std'].objs_[0]

In [ ]:
e2.add_trainer('trainer')

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import ShuffleSplit, KFold
from sklearn.tree import DecisionTreeClassifier
def create_sample_data():
    np.random.seed(42)
    n = 100
    return pd.DataFrame({
        'f1': np.random.randn(n),
        'f2': np.random.randn(n),
        'f3': np.random.randn(n),
        'target': np.random.randint(0, 2, n),
    })
def create_exp(tmp_path, sample_data):
    e = Experimenter(
        data=sample_data,
        path=tmp_path + '/exp',
        sp=ShuffleSplit(n_splits=2, test_size=0.2, random_state=42),
        sp_v=KFold(n_splits=3, shuffle=True, random_state=42),
    )
    e.set_grp('scale', role='stage', processor=StandardScaler,
              method='transform', edges={'X': [(None, ['f1', 'f2', 'f3'])]})
    e.set_node('scaler', grp='scale')
    e.set_grp('model', role='head', processor=DecisionTreeClassifier,
              method='predict',
              edges={'X': [('scaler', None)], 'y': [(None, 'target')]},
              params={'max_depth': 3, 'random_state': 42})
    e.set_node('dt', grp='model')
    e.build()
    e.exp()
    return e

exp = create_exp('tmp', create_sample_data())

In [ ]:
trainer = exp.add_trainer('t1')
trainer.select_head(['dt'])
assert trainer.get_n_splits() == 3
trainer.train()
objs = list(trainer.node_objs['dt'].get_obj())
assert len(objs) == 3